# 워크숍 4 — 에이전트: 도구 호출로 LLM 제어하기

**예상 소요 시간:** 약 90분  
**선수 학습:** 워크숍 1 (워크숍 2와 3도 도움이 되지만 필수는 아닙니다)

이 워크숍에서는 다음 내용을 학습합니다.
1. ReAct(Reason + Act, 추론 + 행동) 에이전트 루프 이해하기
2. 로봇을 제어하는 도구 호출형 LLM 에이전트 만들기
3. 새로운 도구 두 가지, `get_scene_summary`와 `check_held_object` 구현하기
4. 에이전트에서 자주 발생하는 실패 원인 디버깅하기
5. 더 풍부한 도구, 비전 전용 에이전트, 다중 에이전트 파이프라인으로 확장해 보기

> ⚠️ **시작하기 전에:** 이전 워크숍에서 열어 둔 시뮬레이터 창을 모두 닫으세요. 각 노트북은 새로운 로봇 인스턴스를 생성합니다. 뷰어 탭을 두 개 동시에 열면 어떤 로봇을 보고 있는지 혼동될 수 있습니다.

## 섹션 1 — 빠른 설정

In [ ]:
%pip install -q "menlo-robot-sdk[livekit]" python-dotenv requests

In [ ]:
import asyncio
import os
import re
import json
import math
import base64
import time
import requests

from menlo_robot_sdk import AsyncClient, connect
from menlo_robot_sdk.experimental import generate_room_key

# ── 키 로더: 먼저 .env를 확인한 다음 Colab Secrets를 확인합니다 ───────────────
def _load_keys():
    # 모드 B: 로컬 .env 파일
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv가 설치되지 않았으므로 .env 모드가 아닙니다

    # 모드 A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # 키가 이미 환경 변수에 설정되어 있습니다(CI, Docker 등)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://api.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")
print(f"MENLO_API_KEY    : {MENLO_API_KEY[:12]}...")
print(f"TOKAMAK_API_KEY  : {'(set)' if TOKAMAK_API_KEY else '(not set — needed for the agent LLM)'}")

In [ ]:
# 워크숍 1과 동일한 설정
client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

r = await client.robots.create(name=f"workshop4-{int(time.time())}", model="asimov-v0")
robot_id = r.robot.id
await client.robots.create_session(robot_id)

room_key = await generate_room_key(client, robot_id)
print(f"VIEWER URL:\n{VIEWER_BASE_URL}/?key={room_key}")

> 뷰어 URL을 **Google Chrome**에서 열고 장면이 로드될 때까지 기다린 다음, 다음 셀을 실행하세요.

In [ ]:
session = await connect(
    client, robot_id,
    worker_names=[], rcw_identity_prefix="simplesim", join_livekit=True,
)
print(f"Connected: {session.session.room_name}")


async def wait_for_skills(timeout_s: float = 180.0):
    """뷰어가 접속하고 스킬을 사용할 수 있을 때까지 주기적으로 확인합니다."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # 아직 뷰어가 접속하지 않았습니다
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print(f"Skills: {[s.name for s in skills]}")

## 섹션 2 — 도구 호출형 에이전트가 필요한 이유

직접 작성한 로봇 제어 로직은 규모가 커질수록 한계에 부딪힙니다. 상태와 작업의 수가 늘어나면 if/else 분기의 수가 기하급수적으로 증가하기 때문입니다. LLM 에이전트는 이 문제에 다른 방식으로 접근합니다.

### ReAct 루프

```
사용자 작업
   ↓
[LLM 추론] → JSON 도구 호출
   ↓
[실행] → SDK 호출 → 로봇 동작
   ↓
[관찰] → 결과 문자열을 대화 기록에 추가
   ↓
[LLM 추론] → 다음 호출 또는 "done"
```

**핵심 특징:**
- LLM은 텍스트만 볼 수 있으므로 로봇 상태를 텍스트(도구 실행 결과)로 표현합니다.
- 상태를 저장하지 않는 API이므로 매 요청에 전체 대화 기록을 포함합니다.
- 무한 루프를 막기 위해 에이전트의 최대 턴 수를 제한합니다.
- 도구는 언어와 물리 세계를 연결하는 인터페이스입니다.

## 섹션 3 — LLM 및 VLM 도우미 함수

In [ ]:
TOKAMAK_URL = "https://api.tokamak.sh/v1/chat/completions"

def call_llm(messages, model="minimaxai/minimax-m2.7"):
    """Tokamak LLM API에 메시지를 보내고 텍스트 응답을 반환합니다."""
    response = requests.post(
        TOKAMAK_URL,
        headers={
            "Authorization": f"Bearer {TOKAMAK_API_KEY}",
            "Content-Type": "application/json",
        },
        json={"model": model, "messages": messages},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


def ask_vlm(jpeg_bytes, prompt, model="qwen/qwen3.6-35b-a3b"):
    """이미지와 텍스트 프롬프트를 비전-언어 모델에 전송합니다."""
    b64_image = base64.b64encode(jpeg_bytes).decode("utf-8")
    image_url = f"data:image/jpeg;base64,{b64_image}"
    messages = [{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": image_url}},
            {"type": "text", "text": prompt},
        ],
    }]
    return call_llm(messages, model=model)


# 간단한 테스트
reply = call_llm([{"role": "user", "content": "Say 'LLM ready' and nothing else."}])
print(f"LLM test: {reply}")

## 섹션 4 — 도구 레지스트리

도구 레지스트리는 도구 이름을 해당 설명과 연결합니다. LLM은 이 설명을 읽고 어떤 도구를 호출할지 결정합니다.

**좋은 도구 설명의 조건:**
- 단위가 포함된 정확한 인자 이름 (예: `vx in m/s, max 1.5`)
- 명확한 사전 조건 (예: `must be within 1m of target`)
- 예상 반환값 (예: `returns cube id and color`)
- 알려진 제약이나 특이사항 (예: `cannot spin in place`)

In [ ]:
TOOLS = {
    "set_velocity": {
        "description": (
            "Move the robot for a fixed duration. "
            "Args: vx (forward m/s, max 1.5), vy (sideways m/s), "
            "wz (turn rad/s, max 0.6), duration_s. "
            "IMPORTANT: cannot spin in place — always pair wz with vx=0.2."
        )
    },
    "go_to": {
        "description": (
            "Walk to a named entity using built-in pathfinding. "
            "Args: entity_id (e.g. 'pad_C', 'cube_2'). "
            "Pads: pad_A, pad_B, pad_C, pad_D, pad_E."
        )
    },
    "pick_cube": {
        "description": (
            "Pick up the nearest visible cube on the floor. "
            "No args. Returns the cube id and its color. "
            "Precondition: must be within ~1m of a cube (use go_to first)."
        )
    },
    "look": {
        "description": (
            "Capture the robot's POV camera and describe what it sees. "
            "No args. Returns a text description of the scene."
        )
    },
    "place": {
        "description": (
            "Deliver the currently held cube to its correct pad. "
            "Routing rules: red→pad_B, green→pad_C, blue→pad_D, yellow→pad_E. "
            "No args — routing is automatic based on held cube color. "
            "WARNING: delivering to the wrong pad ends the benchmark round."
        )
    },
    "get_scene_summary": {
        "description": (
            "Get a structured summary of all visible cubes from scene state: "
            "their IDs, colors, and distances from the robot. "
            "No args. Use this to plan which cube to pick next."
        )
    },
    "check_held_object": {
        "description": (
            "Check what the robot is currently holding. "
            "No args. Returns the held entity id and color, or 'nothing' if hands are empty."
        )
    },
}

print(f"Tool registry: {list(TOOLS.keys())}")

## 섹션 5 — 시스템 프롬프트 생성기

시스템 프롬프트는 LLM을 위한 사용 설명서입니다. 작업 형식과 도구 호출 규칙을 설명하고, 사용할 수 있는 모든 도구를 나열합니다.

In [ ]:
def build_system_prompt(tools):
    """모든 도구를 JSON 호출 형식으로 안내하는 시스템 프롬프트를 만듭니다."""
    lines = [
        "You control a humanoid warehouse robot by calling tools.",
        "To call a tool, reply with ONLY a single JSON object in a fenced code block:",
        "```json",
        '{"tool": "<tool name>", "args": {...}}',
        "```",
        "Write nothing else outside the code block.",
        "When the task is complete (or impossible), call the special tool 'done':",
        '{"tool": "done", "args": {"summary": "<one sentence>"}}',
        "",
        "Available tools:",
    ]
    for name, spec in tools.items():
        lines.append(f"- {name}: {spec['description']}")
    return "\n".join(lines)


# 모델에 전달되는 내용을 확인할 수 있도록 프롬프트를 출력합니다
print(build_system_prompt(TOOLS))

> **팁:** 시스템 프롬프트의 작은 표현 차이가 에이전트의 행동을 크게 바꿀 수 있습니다. 도구 설명을 조금 수정한 뒤 에이전트의 전략이 어떻게 달라지는지 관찰해 보세요.

## 섹션 6 — 도구 호출 파서

In [ ]:
def parse_tool_call(reply):
    """모델 응답에서 JSON 도구 호출을 추출해 파싱합니다."""
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", reply, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    raise ValueError(f"Could not parse tool call from: {reply[:200]}")


def safe_parse_tool_call(reply):
    """parse_tool_call과 같지만 예외를 발생시키는 대신 오류 딕셔너리를 반환합니다."""
    try:
        return parse_tool_call(reply)
    except Exception as e:
        return {"tool": "error", "args": {"message": str(e), "raw_reply": reply[:200]}}


# 올바른 응답 예시
valid_reply = '```json\n{"tool": "go_to", "args": {"entity_id": "pad_C"}}\n```'
print("Valid:", parse_tool_call(valid_reply))

# 올바르지 않은 응답 예시
bad_reply = "I think we should go to pad_C. Let me navigate there."
print("Invalid:", safe_parse_tool_call(bad_reply))

## 섹션 7 — 도구 실행기

실행기는 각 도구 호출을 실제 SDK로 전달합니다. 즉, 언어 계층과 물리 로봇 사이를 연결합니다.

In [ ]:
async def execute_tool(name, args):
    """도구 호출을 실행하고 텍스트 결과 문자열을 반환합니다."""

    if name == "set_velocity":
        result = await session.invoke("set_velocity", args, timeout_s=60)
        return f"set_velocity finished: status={result.status}"

    if name == "go_to":
        result = await session.invoke(
            "go_to",
            {"target": {"kind": "entity", "entity_id": args["entity_id"]}},
            timeout_s=300,
        )
        if result.status != "done":
            err = f" ({result.error.message})" if result.error else ""
            return f"go_to failed{err}"
        return f"Reached {args['entity_id']}"

    if name == "look":
        jpeg = await session.get_vision("pov")
        return ask_vlm(jpeg, "Describe what you see: objects, colors, positions, anything relevant for warehouse sorting.")

    if name == "pick_cube":
        scene  = await session.state.get("scene_state")
        status = await session.state.get("robot_status")
        rx, ry = status.robot.pose.position[0], status.robot.pose.position[1]
        visible = [
            (eid, e) for eid, e in scene.entities.items()
            if eid.startswith("cube_") and e.visible and e.attached_to is None
        ]
        if not visible:
            return "No visible cubes to pick."
        nearest_id, nearest_e = min(
            visible,
            key=lambda x: math.hypot(x[1].pose.position[0]-rx, x[1].pose.position[1]-ry)
        )
        result = await session.invoke(
            "pick_entity",
            {"target": {"kind": "entity", "entity_id": nearest_id}},
            timeout_s=300,
        )
        if result.status != "done":
            return f"pick_entity failed: {result.error.message if result.error else 'unknown'}"
        color = nearest_e.state.get("color", "unknown")
        return f"Picked {nearest_id} ({color})"

    if name == "place":
        status = await session.state.get("robot_status")
        scene  = await session.state.get("scene_state")
        held = [
            eid for eid, e in scene.entities.items()
            if e.attached_to == status.robot.entity_id
        ]
        if not held:
            return "Not holding anything."
        cube_color = scene.entities[held[0]].state.get("color", "")
        PAD_MAP = {"red": "pad_B", "green": "pad_C", "blue": "pad_D", "yellow": "pad_E"}
        pad_id = PAD_MAP.get(cube_color)
        if not pad_id:
            return f"Unknown color '{cube_color}' — cannot determine pad."
        result = await session.invoke(
            "place_entity",
            {"target": {"kind": "entity", "entity_id": pad_id}},
            timeout_s=300,
        )
        if result.status != "done":
            return "place_entity failed"
        return f"Placed {cube_color} cube on {pad_id}"

    if name == "get_scene_summary":
        scene  = await session.state.get("scene_state")
        status = await session.state.get("robot_status")
        rx, ry = status.robot.pose.position[0], status.robot.pose.position[1]
        lines = []
        for eid, e in scene.entities.items():
            if eid.startswith("cube_") and e.visible and e.attached_to is None:
                dist  = math.hypot(e.pose.position[0]-rx, e.pose.position[1]-ry)
                color = e.state.get("color", "unknown")
                lines.append(f"  {eid}: {color}, {dist:.1f}m away")
        return ("Visible cubes:\n" + "\n".join(lines)) if lines else "No visible cubes."

    if name == "check_held_object":
        status = await session.state.get("robot_status")
        scene  = await session.state.get("scene_state")
        held = [
            eid for eid, e in scene.entities.items()
            if e.attached_to == status.robot.entity_id
        ]
        if not held:
            return "Holding: nothing"
        color = scene.entities[held[0]].state.get("color", "unknown")
        return f"Holding: {held[0]} ({color})"

    if name == "done":
        return f"[done] {args.get('summary', '')}"

    if name == "error":
        return f"[parse error] {args.get('message', '')}"

    return f"ERROR: unknown tool '{name}'"


print("execute_tool defined.")

## 섹션 8 — 에이전트 루프

In [ ]:
async def run_agent(task, max_turns=8):
    """주어진 작업에 대해 ReAct 에이전트 루프를 실행합니다."""
    messages = [
        {"role": "system",  "content": build_system_prompt(TOOLS)},
        {"role": "user",    "content": task},
    ]
    tool_log = []

    for turn in range(1, max_turns + 1):
        reply = call_llm(messages)                      # 추론(THINK)
        messages.append({"role": "assistant", "content": reply})

        call      = safe_parse_tool_call(reply)
        tool_name = call["tool"]
        tool_args = call.get("args", {})

        history_chars = sum(len(str(m["content"])) for m in messages)
        print(f"turn {turn} | tool={tool_name} | args={tool_args} | history~{history_chars:,}chars")

        if tool_name == "done":
            print(f"\n✓ Agent done: {tool_args.get('summary', '')}")
            break

        result_text = await execute_tool(tool_name, tool_args)  # 행동(ACT)
        tool_log.append({"turn": turn, "tool": tool_name, "result": result_text[:100]})
        print(f"  → {result_text[:120]}")

        messages.append({"role": "user", "content": result_text})  # 관찰(OBSERVE)

    return messages, tool_log


# 실행 예시
print("Running agent: pick and deliver the nearest cube...")
messages, tool_log = await run_agent(
    "Use get_scene_summary to find a cube, go to it, pick it up, and deliver it to the correct pad.",
    max_turns=10,
)

## 섹션 9 — 에이전트 디버깅

에이전트가 예상과 다르게 행동하면 전체 메시지 기록과 도구 호출 로그를 살펴보세요.

In [ ]:
# 마지막 실행의 전체 메시지 기록을 출력합니다
print(f"Total messages: {len(messages)}")
for i, m in enumerate(messages):
    role    = m["role"]
    content = str(m["content"])
    preview = content[:120].replace("\n", " ")
    print(f"  [{i}] {role}: {preview}")

In [ ]:
# 도구 호출 로그를 표 형태로 출력합니다
print(f"{'Turn':<6} {'Tool':<22} {'Result (truncated)'}")
print("-" * 70)
for entry in tool_log:
    print(f"  {entry['turn']:<4} {entry['tool']:<22} {entry['result'][:40]}")

### 에이전트에서 자주 발생하는 실패 유형

| 증상 | 원인 | 해결 방법 |
|---|---|---|
| 에이전트가 존재하지 않는 엔티티 ID로 `go_to`를 호출함 | 모델이 엔티티 이름을 지어냄(환각) | `get_scene_summary`를 추가하고 가장 먼저 호출하도록 지시합니다. |
| 에이전트가 같은 도구를 계속 반복 호출함 | 작업 완료 여부를 판단할 신호가 없음 | 최대 턴 수를 정하고, 이미 끝난 작업이라면 도구 결과에 "already done"을 포함합니다. |
| 에이전트가 잘못된 패드로 물체를 옮김 | 모델이 색상→패드 규칙을 모름 | `place` 도구 설명에 경로 규칙을 명시합니다. |
| `parse_tool_call`이 실패함 | 모델이 JSON 대신 자유 형식 텍스트를 생성함 | `safe_parse_tool_call`을 추가하고 시스템 프롬프트에서 JSON만 출력하라는 제약을 강조합니다. |

---
## 연습 문제

### 연습 문제 1 — `spin` 도구 추가하기

로봇을 360° 완전히 회전시키는 `spin` 도구를 `TOOLS`에 추가하세요.
- `set_velocity(wz=0.5, vx=0.2, duration_s=12.5)`는 0.5 rad/s에서 약 360° 회전합니다.

`execute_tool`에 이 도구를 구현한 다음, 에이전트에게 "회전하며 주변 장면을 살펴본 뒤 어떤 색상의 큐브가 보이는지 알려줘."라는 작업을 실행해 보세요.

In [ ]:
## 연습 문제 1: spin 도구 추가하기

# 1. TOOLS에 추가하세요:
# TOOLS["spin"] = {"description": "..."}

# 2. execute_tool에 추가하세요(또는 새 버전을 정의하세요):
# if name == "spin":
#     ...

# 3. 에이전트를 실행하세요:
# 여기에 코드를 작성하세요

# 예상 동작: 에이전트가 spin, look, done 순서로 호출합니다
# 출력에는 360° 탐색 후 보이는 큐브 색상에 대한 설명이 포함됩니다

### 연습 문제 2 — 시스템 프롬프트에 패드 지도 추가하기

각 패드의 이름과 대략적인 x, y 위치를 나열하는 "Map of pads:" 섹션을 `build_system_prompt` 뒤에 추가하세요. 위치는 `scene_state`에서 가져올 수 있습니다. 이렇게 하면 에이전트의 탐색 실수가 줄어드는지 확인해 보세요.

In [ ]:
## 연습 문제 2: 시스템 프롬프트에 패드 지도 추가하기

# 1. 패드 위치를 읽으세요:
# scene = await session.state.get("scene_state")
# pad_map_text = "Map of pads:\n"
# for eid, e in scene.entities.items():
#     if eid.startswith("pad_"): ...

# 2. pad_map_text를 덧붙이도록 build_system_prompt를 수정하세요

# 3. 다음 작업으로 에이전트를 실행하세요: "Sort all visible cubes to their correct pads."

# 여기에 코드를 작성하세요

# 예상 동작: 에이전트가 패드 위치를 사용해 더 효율적으로 이동합니다

### 연습 문제 3 — 비전 전용 도구

OpenCV 기반 HSV 분할을 사용하는 `perceive` 도구를 추가하세요(`scene_state`는 사용하지 않습니다). `perceive()` 함수를 셀 안에 복사해 넣으세요.

비전 관련 도구만 남긴 `{"set_velocity", "perceive", "pick_cube", "place", "done"}` 구성으로 에이전트를 실행하세요. `go_to`와 `get_scene_summary`는 제거합니다. 무엇이 제대로 작동하지 않는지 관찰해 보세요.

In [ ]:
## 연습 문제 3: 비전 전용 에이전트
import cv2
import numpy as np

# 워크숍 2의 perceive()를 여기에 복사하세요:
async def perceive(session):
    # 여기에 코드를 작성하세요(워크숍 2의 perceive() 함수 전체를 붙여 넣으세요)
    pass

# TOOLS에 perceive를 추가하고 go_to와 get_scene_summary를 제거하세요
# TOOLS_VISION = {k: v for k, v in TOOLS.items() if k not in ("go_to", "get_scene_summary")}
# TOOLS_VISION["perceive"] = {"description": "..."}

# 여기에 코드를 작성하세요

# 예상 동작: 에이전트가 작업을 시도하지만 go_to가 없어 어려움을 겪을 수 있습니다
# 관찰: 무엇을 다르게 하나요? 성공하나요? 가장 먼저 실패하는 부분은 무엇인가요?

### 연습 문제 4 — 에이전트 루프 계측하기

도구 종류별 호출 횟수를 세도록 `run_agent`를 수정하거나 래퍼 함수를 작성하세요. 실행 후에는 다음과 같은 요약 표를 출력합니다.
```
go_to:              3 calls
pick_cube:          2 calls
get_scene_summary:  1 call
place:              2 calls
done:               1 call
```

In [ ]:
## 연습 문제 4: 도구 종류별 호출 횟수 세기
from collections import Counter

# 여기에 코드를 작성하세요

# 예상 출력(예시):
# get_scene_summary:  1 calls
# go_to:              3 calls
# pick_cube:          2 calls
# place:              2 calls
# done:               1 calls

### 연습 문제 5 — 축소된 도구 모음

도구 레지스트리에서 `go_to`와 `get_scene_summary`를 제거하고 `set_velocity`, `pick_cube`, `look`, `place`, `check_held_object`, `done`만 남기세요. 분류 작업을 세 번 실행하고 성공/실패 여부를 기록하세요.

In [ ]:
## 연습 문제 5: 축소된 도구 모음 — 3회 실행
TOOLS_DEGRADED = {k: v for k, v in TOOLS.items() if k not in ("go_to", "get_scene_summary")}

# 여기에 코드를 작성하세요 — 세 번 실행하고 결과를 기록하세요

# 결과 표:
# 실행 1: [성공 / 실패 / 일부 성공]
# 실행 2: ...
# 실행 3: ...
# 관찰 결과: ...

### 연습 문제 6 (도전) — 두 에이전트 파이프라인

계획자(Planner) + 실행자(Executor) 파이프라인을 만드세요.
1. **계획자:** 장면 요약과 함께 `call_llm`을 호출하고, `[{"action": "go_to", "target": "cube_2"}, {"action": "pick"}, {"action": "go_to", "target": "pad_C"}, {"action": "place"}, ...]`와 같은 JSON 단계 목록을 반환하도록 요청합니다.
2. **실행자:** 계획의 각 단계를 순회하며 SDK를 직접 호출합니다(LLM은 사용하지 않습니다).

이 구조는 상위 수준 계획(LLM의 강점)과 안정적인 실행(SDK의 강점)을 분리합니다.

In [ ]:
## 연습 문제 6(도전): 두 에이전트 계획자 + 실행자

# --- 계획자(Planner) ---
async def make_plan(session):
    """장면 요약을 바탕으로 단계별 JSON 계획을 만들도록 LLM에 요청합니다."""
    # 1. 장면 요약 텍스트를 가져옵니다
    # 2. JSON 단계 목록을 요청하는 프롬프트를 만듭니다
    # 3. JSON을 파싱합니다
    # 여기에 코드를 작성하세요
    pass


# --- 실행자(Executor) ---
async def execute_plan(session, plan):
    """계획의 각 단계를 SDK 직접 호출로 실행합니다."""
    for step in plan:
        action = step.get("action")
        target = step.get("target", "")
        print(f"Executing: {action} {target}")
        # 여기에 코드를 작성하세요 — go_to, pick, place 동작을 처리합니다


# 실행:
# plan = await make_plan(session)
# print("Plan:", plan)
# await execute_plan(session, plan)

---
## 정리

In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")